GET DATA

In [ ]:
import requests, pandas as pd

r = requests.get('https://api.figshare.com/v2/articles/1031637').json()
for f in r['files']:
    print(f['name'], f['size'], f['download_url'])

url = r['files'][0]['download_url']
open('BradleyMeltingPointDataset.xlsx', 'wb').write(requests.get(url).content)
df = pd.read_excel('BradleyMeltingPointDataset.xlsx')
print(df.columns)

BradleyMeltingPointDataset.xlsx 2225265 https://ndownloader.figshare.com/files/1503990
Index(['key', 'name', 'smiles', 'mpC', 'csid', 'link', 'source', 'donotuse',
       'donotusebecause'],
      dtype='object')


Create table for notes

In [ ]:
log = []
def note(col, problem, n_rows, action, reason, status='todo'):
    log.append({'column': col, 'problem': problem,
                'n_rows':int(n_rows),
                'action':action, 'reason':reason, 'status':status})
pd.DataFrame(log).to_csv('decisions_log.csv', index=False)
pd.DataFrame(log)

""


# **Block 1**
Check shape, memory, info,none, categorical and numerical data. Cleaning data

**Shape and memory**

-number of rows

-number of columns

-will you have problem with memory

In [ ]:
df.shape

(28645, 9)

In [ ]:
print(df.memory_usage(deep=True).sum()/ 1e6,'MB')

12.422262 MB


**Info**

-how many numerical columns

-categorical columns

-types

-check all columns and their types

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28645 entries, 0 to 28644
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   key              28645 non-null  int64  
 1   name             28645 non-null  object 
 2   smiles           28645 non-null  object 
 3   mpC              28645 non-null  float64
 4   csid             28645 non-null  int64  
 5   link             28634 non-null  object 
 6   source           28645 non-null  object 
 7   donotuse         377 non-null    object 
 8   donotusebecause  377 non-null    object 
dtypes: float64(1), int64(2), object(6)
memory usage: 2.0+ MB


# **Find none**

-precentage of nones

-check neccessary columns

In [ ]:
print(df.isna().mean().sort_values(ascending=False))

donotuse           0.986839
donotusebecause    0.986839
link               0.000384
key                0.000000
name               0.000000
csid               0.000000
mpC                0.000000
smiles             0.000000
source             0.000000
dtype: float64


In [ ]:
n_rows = df[df['donotuse'].notna()]['donotuse'].count()
note('donotuse', 'bad data',n_rows, 'delete', 'we cannot use it')
n_rows = df[df['donotusebecause'].notna()]['donotusebecause'].count()
note('donotusebecause', 'bad data',n_rows, 'delete', 'we cannot use it')

In [ ]:
pd.DataFrame(log).to_csv('decisions_log.csv', index=False)
pd.DataFrame(log)

,column,problem,n_rows,action,reason,status
0,donotuse,bad data,377,delete,we cannot use it,todo
1,donotusebecause,bad data,377,delete,we cannot use it,todo


In [ ]:
n_rows= len(df['link'])
note('donotuse, donotusebecause, link, source', 'not need',n_rows, 'delete', 'it donot use for the models')
pd.DataFrame(log).to_csv('decisions_log.csv', index=False)

Apply some notes

In [ ]:
df = df[df['donotusebecause'].isna()].copy()
df_m = df[['key' ,'name','smiles','mpC','csid']].copy()

**Categorical data**

-smiles cannonic

-what in each columns

Should lead smiles to canonic


In [ ]:
!pip install rdkit

In [ ]:
from rdkit import Chem

def can(s):
  if not isinstance(s,str):
    return None
  m = Chem.MolFromSmiles(s)
  return Chem.MolToSmiles(m) if m else None

In [ ]:
df_m['smiles_can'] = df_m['smiles'].apply(can)

In [ ]:
df_m[df_m['smiles_can'].isna()]

,key,name,smiles,mpC,csid,smiles_can
12469,12470,(diphenylmethylidene)(triphenyl)-lambda~5~-pho...,[P+](c1ccccc1)(c1ccccc1)(c1ccccc1)C(=C=1C=CC=C...,170.0,436055,None
14776,14777,(2'beta)-22-Oxovincaleukoblastine,O=C(OC)[C@@]2(C[C@H]4C[C@@](O)(CC)C[N@@](CCc1c...,219.0,5758,None
14831,14832,"(8beta)-N,N-Diethyl-6-methyl-9,10-didehydroerg...",CCN(CC)C(=O)[C@@H]2C=C1c3cccc4ncc(C[C@H]1N(C)C...,82.5,5558,None
14948,14949,"1,3-Dimethyl-3,7-dihydro-1H-purine-2,6-dione",O=C2c1ncnc1N(C)C(=O)N2C,273.0,2068,None
15029,15030,"1H-1,2,4-Triazol-3-amine",Nc1ncnn1,159.0,1577,None
...,...,...,...,...,...,...
27528,27529,2-methylbenzimidazole,n2c1ccccc1nc2C,176.5,11489,None
27606,27607,3-indoleacrylic acid,O=C(O)\C=C\c2c1ccccc1nc2,185.0,4524636,None
27719,27720,6-mercaptopurine,S=C2/N=C\Nc1ncnc12,314.0,580869,None
28189,28190,indole-3-acetic acid,O=C(O)Cc2c1ccccc1nc2,168.5,780,None


In [ ]:
n_rows = len(df_m[df_m['smiles_can'].isna()]['smiles_can'])
note('smiles_can', 'isnot able to creat smiles non-canonical ', n_rows, 'check', 'are able to recover')

In [ ]:
pd.DataFrame(log).to_csv('decisions_log.csv', index=False)

Try to recover none smiles_can from name

In [ ]:
!pip install pubchempy

In [ ]:
import pubchempy as pcp, time

def smiles_from_name(name):
  try:
    c = pcp.get_compounds(name, 'name')
    time.sleep(0.25)
    return c[0].isomeric_smiles if c else None
  except Exception:
    return None

bad = df_m[df_m['smiles_can'].isna()].copy()
bad['smiles_pc'] = bad['name'].apply(smiles_from_name)
bad['smiles_can'] = bad['smiles_pc'].apply(can)

recovered = bad.dropna(subset=['smiles_can'])
print('recovery:', len(recovered), 'from', len(bad))

/tmp/ipykernel_1201/1294445578.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return c[0].isomeric_smiles if c else None


recovery: 261 from 292


In [ ]:
print(recovered[["name",'key', "smiles_can", "mpC"]].sample(20, random_state=0).to_string())

                               name    key                                                                                                                                           smiles_can    mpC
22087                    carvedilol  22088                                                                                                          COc1ccccc1OCCNCC(O)COc1cccc2[nH]c3ccccc3c12  114.5
18920     2-methyl-5-nitroimidazole  18921                                                                                                                            Cc1ncc([N+](=O)[O-])[nH]1  253.0
25369           2-mercaptoimidazole  25370                                                                                                                                      S=c1[nH]cc[nH]1  229.5
17097                   allopurinol  17098                                                                                                                                O=c1[nH]cnc2[nH]ncc12  350.0
24474

In [ ]:
n_rows = 31
note('smiles_can', 'isnot able to creat smiles non-canonical ', n_rows, 'delete', 'dont recovery smiles')

In [ ]:
good = df_m.dropna(subset=["smiles_can"]).copy()
good["source"] = "original"
recovered["source"] = "pubchem_by_name"
df_final = pd.concat([good, recovered], ignore_index=True)
print(len(good), len(recovered), "->", len(df_final))

27976 261 -> 28237


/tmp/ipykernel_1201/3252915713.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recovered["source"] = "pubchem_by_name"


# **Numerical data**

In [ ]:
df_final.describe()

,key,mpC,csid
count,28237.000000,28237.000000,2.823700e+04
mean,14291.124942,94.488917,2.367572e+06
std,8286.061151,96.615066,5.117577e+06
min,1.000000,-205.000000,0.000000e+00
25%,7073.000000,35.000000,1.048700e+04
50%,14317.000000,97.000000,6.684700e+04
75%,21462.000000,163.000000,2.031618e+06
max,28645.000000,517.000000,2.829647e+07


In [84]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/melting_point_project"   # поправьте под своё название папки
RAW = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

df_final.to_csv(f"{PROC}/melting_points_clean.csv", index=False)

In [85]:
meta = pd.read_csv(f"{PROC}/melting_points_clean.csv")

In [ ]:
meta['smiles_can'].value_counts()

,count
smiles_can,
O=C(O)c1ccccc1O,16
O=C(O)c1ccccc1,14
O=C(O)C(Cl)Cl,14
CC(=O)Nc1ccc(O)cc1,13
CC(=O)Oc1ccccc1C(=O)O,13
...,...
CC[C@@H]1CN2CCc3c([nH]c4cccc(OC)c34)[C@@H]2C[C@@H]1/C(=C\OC)C(=O)OC,1
Cc1nccc2c1[nH]c1ccccc12,1
CC[C@H]1C[C@@H]2C[C@H]3c4[nH]c5cc(OC)ccc5c4CCN(C2)[C@@H]13,1


In [88]:
MAX_SPREAD = 20      # К

agg = df_final.groupby("smiles_can").agg(
    mpC=("mpC", "median"),
    spread_K=("mpC", lambda s: s.max() - s.min()),
).reset_index()

agg = agg[agg["spread_K"] <= MAX_SPREAD].reset_index(drop=True)
agg["mpC"] = agg["mpC"] + 273.15

clean = agg[["smiles_can", "mpC"]]
clean.to_csv(f"{PROC}/melting_points_clean.csv", index=False)

print("num molecules:", len(clean))
print(clean.isna().sum())

num molecules: 19824
smiles_can    0
mpC           0
dtype: int64
